# What extraction costs, per model and per design — before spending anything

The numbers behind the model choice (Fase 4): how long contracts are in CUAD, and what one
extraction costs with each candidate model under the two context designs.

- **D2**: each block call reads the skeleton (parties, clause outline, definitions) plus the
  block and its neighbours.
- **D1**: each block call reads the skeleton plus the whole contract, cached after the first call.

Prices come from `services/llm/cost_estimator.py`, the same table the provider bills with.
Output per contract is taken as 2.2 tokens per contract token, from the six graphs built by
the first pipeline. **Reasoning tokens are not included**: gpt-6.1-sol always reasons, and its
real cost is measured in `runs.json`, not here.

In [1]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'server').exists():
    ROOT = ROOT.parent
SERVER = ROOT / 'server'
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))

import math
from services.llm.cost_estimator import LONG_CONTEXT_THRESHOLD, MODEL_PRICING_USD_PER_1M

## How long contracts are

Tokens are approximated as characters / 4. The 272k threshold of long-context billing is per
request, and only design D1 ever sends a whole contract in one request.

In [2]:
cuad = json.loads((ROOT / 'infra/json/CUAD_v1.json').read_text())
docs = cuad['data'] if isinstance(cuad, dict) and 'data' in cuad else cuad
lengths = sorted((len(''.join(p['context'] for p in d['paragraphs'])) // 4, d['title']) for d in docs)
tokens = [n for n, _ in lengths]
n = len(tokens)
print(f'{n} CUAD contracts | median {tokens[n // 2]:,} | p90 {tokens[int(n * .9)]:,} | p99 {tokens[int(n * .99)]:,} | max {tokens[-1]:,} tokens')
for threshold in (50_000, 100_000, LONG_CONTEXT_THRESHOLD):
    print(f'  longer than {threshold:>7,}: {sum(t > threshold for t in tokens)}')

ours = {}
for path in sorted((ROOT / 'infra/json/paragraphs').glob('*.json')):
    paragraphs = json.loads(path.read_text())['paragraphs']
    size = sum(len(p['text']) for p in paragraphs) // 4
    ours[path.stem[5:25]] = size
    print(f'  {path.stem[5:45]:42s} {size:>7,} tokens  (CUAD percentile {100 * sum(t <= size for t in tokens) // n})')

510 CUAD contracts | median 8,356 | p90 30,672 | p99 68,004 | max 84,552 tokens
  longer than  50,000: 15
  longer than 100,000: 0
  longer than 272,000: 0
  BELLICUMPHARMACEUTICALS_INC_05_07_2019-E    45,858 tokens  (CUAD percentile 96)
  EdietsComInc_20001030_10QSB_EX-10_4_2606    14,808 tokens  (CUAD percentile 71)
  HealthcentralCom_19991108_S-1A_EX-10_27_     9,982 tokens  (CUAD percentile 57)
  RitterPharmaceuticalsInc_20200313_S-4A_E    29,761 tokens  (CUAD percentile 89)
  SteelVaultCorp_20081224_10-K_EX-10_16_30     4,341 tokens  (CUAD percentile 27)
  TomOnlineInc_20060501_20-F_EX-4_46_74970    27,387 tokens  (CUAD percentile 86)


## Cost of one extraction

Each cell is **D1 / D2**. `no cabe` marks a model whose input limit the request exceeds.

In [3]:
PROMPT = 3400                    # system prompt and output shape, per call
MAX_INPUT = {'gpt-5.4-mini': 272_000}
MODELS = ['gpt-4.1', 'gpt-4.1-mini', 'gpt-6.1-sol', 'gpt-6-sol', 'gpt-6-luna', 'gpt-5.4-mini']

def call_cost(model, new, cached, written, out):
    total = new + cached + written
    if total > MAX_INPUT.get(model, 922_000):
        return None
    pricing = MODEL_PRICING_USD_PER_1M[model]
    rates = pricing.long if pricing.long and total > LONG_CONTEXT_THRESHOLD else pricing.short
    cached_rate = rates.input if rates.cached_input is None else rates.cached_input
    write_rate = rates.input if rates.cache_write is None else rates.cache_write
    return (new * rates.input + cached * cached_rate + written * write_rate + out * rates.output) / 1e6

def extraction_cost(model, doc, design):
    calls = math.ceil(doc / 1250)                     # one block per ~5000 characters
    out = 2.2 * doc / calls
    prefix, new = ((PROMPT + 500 + doc, 200) if design == 'D1' else (PROMPT + 1500 + int(0.05 * doc), 2500))
    total = 0.0
    for k in range(calls):                            # the first call writes the cache, the rest read it
        cost = call_cost(model, new, 0 if k == 0 else prefix, prefix if k == 0 else 0, out)
        if cost is None:
            return None
        total += cost
    return total

sizes = [(ours.get('SteelVaultCorp_20081', 4500), 'SteelVault'), (16_000, 'Ediets'), (47_000, 'Bellicum'),
         (tokens[-1], 'longest CUAD'), (150_000, 'outside CUAD'), (300_000, 'outside CUAD')]
print(f"{'tokens':>8} {'':13}" + ''.join(f'{m:>17}' for m in MODELS))
for doc, label in sizes:
    cells = []
    for model in MODELS:
        d1, d2 = extraction_cost(model, doc, 'D1'), extraction_cost(model, doc, 'D2')
        cells.append(('no cabe' if d1 is None else f'${d1:,.2f}') + ' / ' + f'${d2:,.2f}')
    print(f'{doc:>8,} {label:13}' + ''.join(f'{c:>17}' for c in cells))

  tokens                        gpt-4.1     gpt-4.1-mini      gpt-6.1-sol        gpt-6-sol       gpt-6-luna     gpt-5.4-mini
   4,341 SteelVault       $0.11 / $0.11    $0.02 / $0.02    $0.12 / $0.13    $0.12 / $0.13    $0.01 / $0.01    $0.05 / $0.06
  16,000 Ediets           $0.45 / $0.39    $0.09 / $0.08    $0.43 / $0.44    $0.45 / $0.44    $0.02 / $0.02    $0.19 / $0.19
  47,000 Bellicum         $1.89 / $1.17    $0.38 / $0.23    $1.36 / $1.27    $1.55 / $1.30    $0.08 / $0.06    $0.65 / $0.56
  84,552 longest CUAD     $4.66 / $2.15    $0.93 / $0.43    $2.70 / $2.28    $3.29 / $2.35    $0.16 / $0.12    $1.36 / $1.02
 150,000 outside CUAD    $12.15 / $4.00    $2.43 / $0.80    $5.56 / $4.08    $7.40 / $4.23    $0.37 / $0.21    $2.99 / $1.83
 300,000 outside CUAD    $42.30 / $8.90    $8.46 / $1.78   $26.14 / $8.33   $40.66 / $8.80    $2.03 / $0.44  no cabe / $3.79
